# Preparação do Banco — Setup da Arquitetura Medalhão

Este notebook **deve ser executado primeiro**, antes das pipelines (01-04). Ele prepara o ambiente no Unity Catalog criando os schemas para cada camada do medalhão.

## Objetivo do notebook de preparação

### 1. Reprodutibilidade
O script pode ser executado quantas vezes for necessário — o resultado é sempre o mesmo. O `DROP IF EXISTS CASCADE` remove schemas e tabelas antigas antes de recriar tudo limpo. Isso garante que não restem tabelas "órfãs" de execuções anteriores com schemas desatualizados.

>Basta executar essa pipiline de preparação, em seguida as 3 camadas (01 até 03), e por ultimo é preciso criar um volume na camada bronze e executar a 4 pipiline que sé independente, e usada para carregar dados manualmente e complementa os dados nas 3 camadas.

### 2. Separação de responsabilidades
A **infraestrutura** (catálogo, schemas, permissões) fica separada da **lógica de dados** (extração, transformação, carga). Cada notebook tem uma responsabilidade única:
- `00_preparacao_banco` → infraestrutura
- `01_bronze_bacen` → pipeline de  carga de Reclamações, ouvidorias e consórcios  **(dados da API)**
- `02_silver_bacen` → pipeline limpeza, tratamento de nulls de de Reclamações, ouvidorias e consórcios **(dados da API)**
- `03_gold_bacen` → pipeline agregação, sumarizações, Joins de de Reclamações, ouvidorias e consórcios **(dados da API)**
- `04_pipeline_ingestao_site` → pipeline de ingestão web apartada, e não foi separada em camadas **(dados do site, extração manual, e carregados no volume da bronze)**

### 3. Governança
Schemas separados por camada (bronze, silver, gold) permitem:
- **Controle de acesso granular** — conceder permissão de leitura apenas no Gold para analistas
- **Lineage claro** — visualização do fluxo Bronze → Silver → Gold no Catalog Explorer
- **Organização** — cada tabela fica no schema da sua camada, sem prefixos no nome

## Estrutura criada

| Schema | Camada | Função |
| --- | --- | --- |
| `workspace.bacen_bronze` | Bronze | Dados raw das APIs BACEN (sem transformação) |
| `workspace.bacen_silver` | Silver | Dados limpos, tipados e deduplicados |
| `workspace.bacen_gold` | Gold | Tabelas analíticas prontas para consumo |

> ⚠️ **Atenção**: Este notebook destrói todos os dados existentes nos schemas. Execute apenas quando quiser reconstruir o ambiente do zero. Para cargas incrementais, execute apenas as pipelines (01-04).

**Célula 2** — Criar Schemas Medalhão: explica o DROP CASCADE na ordem inversa (Gold → Bronze) e a recriação dos 3 schemas vazios

**Célula 3**— Comentários UC nos Schemas: descreve que os comentários aparecem no Catalog Explorer e information_schema

**Célula 4** — Verificação dos Schemas: explica a consulta de confirmação no information_schema com nome e comentário de cada schema

In [0]:
%sql
-- ============================================================================
-- PREPARAÇÃO DO BANCO — CRIAÇÃO DE SCHEMAS MEDALHÃO
-- ============================================================================
-- Esta sessão remove todos os schemas legados e do medalhão existentes (com
-- CASCADE, eliminando tabelas e views dependentes) e recria os três schemas
-- vazios: bacen_bronze, bacen_silver e bacen_gold. O DROP segue a ordem
-- inversa (Gold → Silver → Bronze) para respeitar dependências entre camadas.
-- ============================================================================
-- Executa DROP IF EXISTS CASCADE antes de criar, garantindo ambiente limpo.
-- CASCADE remove todas as tabelas dentro do schema automaticamente.
-- ============================================================================

-- Dropar schema antigo (legacy) se existir
DROP SCHEMA IF EXISTS workspace.bacen CASCADE;

-- Dropar schemas existentes (ordem reversa: Gold → Silver → Bronze)
-- para respeitar dependências caso existam views que referenciam outras camadas
DROP SCHEMA IF EXISTS workspace.bacen_gold CASCADE;
DROP SCHEMA IF EXISTS workspace.bacen_silver CASCADE;
DROP SCHEMA IF EXISTS workspace.bacen_bronze CASCADE;

-- Criar schemas da arquitetura medalhão (ordem: Bronze → Silver → Gold)
CREATE SCHEMA workspace.bacen_bronze;
CREATE SCHEMA workspace.bacen_silver;
CREATE SCHEMA workspace.bacen_gold;

In [0]:
%sql
-- ============================================================================
-- COMENTÁRIOS UC — DOCUMENTAÇÃO DOS SCHEMAS
-- ============================================================================
-- Adiciona comentários descritivos em cada schema do Unity Catalog. Esses
-- comentários aparecem no Catalog Explorer e no information_schema, permitindo
-- que outros analistas entendam o propósito de cada camada sem precisar abrir
-- os notebooks da pipeline.
-- ============================================================================
-- Comentários aparecem no Catalog Explorer e em information_schema,
-- facilitando a descoberta e entendimento dos dados por outros analistas.
-- ============================================================================

COMMENT ON SCHEMA workspace.bacen_bronze IS
  'Camada Bronze — dados raw extraídos das APIs BACEN (rdrweb e Olinda). Reclamações, Ouvidorias e Consórcios. Sem transformação, todas as colunas como string.';

COMMENT ON SCHEMA workspace.bacen_silver IS
  'Camada Silver — dados limpos e tipados. Conversão de tipos, tratamento de nulos, padronização de períodos e índices. Deduplicada.';

COMMENT ON SCHEMA workspace.bacen_gold IS
  'Camada Gold — tabelas analíticas prontas para consumo. Rankings, resumos anuais, cruzamentos entre fontes e classificação de risco.';

In [0]:
%sql
-- ============================================================================
-- VERIFICAÇÃO — SCHEMAS CRIADOS
-- ============================================================================
-- Consulta o information_schema para confirmar que os três schemas do
-- medalhão foram criados corretamente no catálogo workspace e que os
-- comentários de documentação foram aplicados. Retorna o nome e o comentário
-- de cada schema.
-- ============================================================================

SELECT schema_name, comment
FROM system.information_schema.schemata
WHERE catalog_name = 'workspace'
  AND schema_name LIKE 'bacen%'
ORDER BY schema_name;